# Rendered without a kernel

Everything below — the plots, the numbers, the widgets — was produced by a
Monte Carlo simulation that ran **once**, while this notebook was being
written. The result was then stored inside the notebook file itself as a
*render cache*.

You are looking at that cache. No Python kernel was downloaded, started, or
executed to show you this page: Specta rebuilt the document straight from the
stored outputs. That is what **static rendering** does.

The simulation takes a few seconds to run. Multiply that by every reader who
ever opens the page — and by the seconds it takes to fetch and boot a
WebAssembly Python kernel first — and it is work worth doing exactly once.


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import HTML, HBox, IntSlider, Layout, Output, VBox

N_PATHS = 20_000
N_STEPS = 250
START = 100.0
DRIFT = 0.06
VOLATILITY = 0.22

rng = np.random.default_rng(20260818)

## The expensive part

Twenty thousand price paths over 250 steps: five million normal draws, plus the
cumulative product that turns them into trajectories.


In [ ]:
started = time.perf_counter()

dt = 1.0 / N_STEPS
shocks = rng.normal(
    loc=(DRIFT - 0.5 * VOLATILITY**2) * dt,
    scale=VOLATILITY * np.sqrt(dt),
    size=(N_PATHS, N_STEPS),
)
paths = START * np.exp(np.cumsum(shocks, axis=1))
paths = np.hstack([np.full((N_PATHS, 1), START), paths])

elapsed = time.perf_counter() - started
print(f"{N_PATHS:,} paths x {N_STEPS} steps simulated in {elapsed:.2f} s")

## Where the paths end up

The shaded bands are the 10–90% and 25–75% ranges across all paths; the solid
line is the median. A handful of individual paths are drawn on top to show how
much any single one wanders away from it.


In [ ]:
steps = np.arange(N_STEPS + 1)
low, q25, median, q75, high = np.percentile(paths, [10, 25, 50, 75, 90], axis=0)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.fill_between(steps, low, high, color="#4c72b0", alpha=0.18, label="10-90%")
ax.fill_between(steps, q25, q75, color="#4c72b0", alpha=0.32, label="25-75%")
ax.plot(steps, median, color="#1f3d7a", lw=2, label="median")
for path in paths[:8]:
    ax.plot(steps, path, color="#c44e52", lw=0.7, alpha=0.6)

ax.set_xlabel("step")
ax.set_ylabel("value")
ax.set_title("20,000 simulated paths")
ax.legend(loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Widgets survive the cache too

The render cache stores the state of every `ipywidgets` model alongside the
outputs, so widgets are rebuilt looking exactly as they did when the cache was
saved. That is what makes static rendering usable for dashboards rather than
only for notebooks full of text and images.


In [ ]:
final = paths[:, -1]

def card(label, value):
    return HTML(
        f"<div style='padding:10px 16px;border:1px solid var(--jp-border-color2);"
        f"border-radius:6px;min-width:130px'>"
        f"<div style='font-size:11px;opacity:0.65;text-transform:uppercase;"
        f"letter-spacing:0.06em'>{label}</div>"
        f"<div style='font-size:22px;font-weight:600'>{value}</div></div>"
    )

HBox(
    [
        card("median", f"{np.median(final):.1f}"),
        card("mean", f"{final.mean():.1f}"),
        card("5th pct", f"{np.percentile(final, 5):.1f}"),
        card("95th pct", f"{np.percentile(final, 95):.1f}"),
        card("above start", f"{(final > START).mean():.0%}"),
    ],
    layout=Layout(flex_flow="row wrap", grid_gap="10px", margin="4px 0 12px 0"),
)

### What a cache cannot do

The slider below is real — it was rendered from stored widget state, and you can
drag it. But the readout next to it only changes when Python is there to answer,
and in a statically rendered page there is no kernel listening.

Open the settings menu in the top bar and press **Render with kernel** to start
one; the slider becomes live, at the cost of the startup you just skipped.


In [ ]:
slider = IntSlider(value=50, min=1, max=99, description="percentile:")
readout = Output()

def show_percentile(change=None):
    readout.clear_output()
    with readout:
        print(f"percentile {slider.value}: {np.percentile(final, slider.value):.2f}")

slider.observe(show_percentile, names="value")
show_percentile()

VBox([slider, readout])

## Distribution of outcomes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(final, bins=80, color="#4c72b0", alpha=0.85)
ax.axvline(START, color="#c44e52", lw=1.5, ls="--", label="start")
ax.axvline(np.median(final), color="#1f3d7a", lw=1.5, label="median")
ax.set_xlabel("value at final step")
ax.set_ylabel("paths")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Refreshing the cache

The cache lives in this notebook's metadata, so it travels with the file — no
sidecar to keep in sync.

To rebuild it, open the notebook in JupyterLab with **Open With ▸ Specta**, wait
for it to finish executing, then press **Save cache** in the settings menu.
Specta stores a hash of the code alongside the cache: edit any cell and the
settings menu will report the cache as out of sync, and offer to re-run the
notebook with a kernel instead of showing you a stale page.
